# PhyloP Conservation Analysis - Complete Workflow

**Complete end-to-end analysis of translon conservation using PhyloP scores**

## Key Features:
- Handles spliced features properly (multi-exon translons)
- Uses **exonic flanks** (walks along transcript exons, skips introns)
- Calculates CDS overlap by proportion
- Identifies translons with restricted conservation

## Outputs:
1. Basic story file (65 restricted candidates + others)
2. Basic column descriptions
3. Comprehensive results (all statistics)
4. Comprehensive column descriptions

In [1]:
# Install required packages
import sys

try:
    import pyBigWig
    import pyranges as pr
    print("All packages already installed")
except ImportError:
    print("Installing required packages...")
    !{sys.executable} -m pip install --user pyBigWig pyranges
    print("\nInstallation complete. Please restart kernel.")
    print("After restart, run this cell again to verify installation.")

All packages already installed


In [2]:
import pandas as pd
import numpy as np
import pyBigWig
import pyranges as pr
import gzip
from collections import defaultdict
from pathlib import Path

print("All imports successful")

All imports successful


# Config


In [3]:
# Input files
BIGBED_URL = 'https://ftp.ebi.ac.uk/pub/databases/gencode/riboseq_orfs/data/Ribo-seq_ORFs.bb'
BIGBED_FILE = 'data/Ribo-seq_ORFs.bb'
TRANSCRIPT_ANNOTATIONS = '../phase1_w_transcript.tsv'
GENCODE_GTF = '../data/gencode.v46.annotation.gtf.gz'
PHYLOP_470WAY = 'data/phylop/hg38.phyloP470way.bw'

# Output directory
OUTPUT_DIR = Path('../results/phylop')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Parameters
FLANK_SIZE = 150  # bases of EXONIC sequence for flanks
CONSERVATION_THRESHOLD = 1.5  # PhyloP threshold for "strong conservation"
FLANK_THRESHOLD = 1.0  # Max flank threshold for "restricted conservation"

print(f"Configuration:")
print(f"  Flank size: {FLANK_SIZE}bp (exonic)")
print(f"  Conservation threshold: {CONSERVATION_THRESHOLD}")
print(f"  Flank threshold: {FLANK_THRESHOLD}")
print(f"  Output directory: {OUTPUT_DIR}")

Configuration:
  Flank size: 150bp (exonic)
  Conservation threshold: 1.5
  Flank threshold: 1.0
  Output directory: ../results/phylop


## Step 1: Load Transcript Exon Structures from GENCODE

In [50]:
print("Loading transcript exon structures from GENCODE GTF...")

transcript_exons = defaultdict(list)

with gzip.open(GENCODE_GTF, 'rt') as f:
    for line in f:
        if line.startswith('#'):
            continue
        
        fields = line.strip().split('\t')
        if len(fields) < 9 or fields[2] != 'exon':
            continue
        
        chrom = fields[0]
        start = int(fields[3]) - 1  # GTF is 1-based, convert to 0-based
        end = int(fields[4])
        strand = fields[6]
        
        # Parse attributes for transcript_id
        attrs = {}
        for attr in fields[8].split(';'):
            attr = attr.strip()
            if attr:
                parts = attr.split(' ', 1)
                if len(parts) == 2:
                    key, val = parts
                    attrs[key] = val.strip('"')
        
        transcript_id = attrs.get('transcript_id', '').split(".")[0]
        if not transcript_id:
            continue
        
        transcript_exons[transcript_id].append({
            'chrom': chrom,
            'start': start,
            'end': end,
            'strand': strand
        })

# Sort exons by start position
for transcript_id in transcript_exons:
    transcript_exons[transcript_id].sort(key=lambda x: x['start'])

print(f"Loaded exons for {len(transcript_exons):,} transcripts")

Loading transcript exon structures from GENCODE GTF...
Loaded exons for 254,070 transcripts


## Step 2: Load Translon Data and Merge with Transcript Annotations

In [51]:
# Check if BigBed file exists, download if needed
import os

if not os.path.exists(BIGBED_FILE):
    print(f"Downloading BigBed file from {BIGBED_URL}...")
    !curl -o {BIGBED_FILE} {BIGBED_URL}
    print(f"Downloaded to {BIGBED_FILE}")
else:
    print(f"BigBed file already exists: {BIGBED_FILE}")

BigBed file already exists: data/Ribo-seq_ORFs.bb


In [52]:
# Convert BigBed to BED
bed_file = BIGBED_FILE.replace('.bb', '.bed')

if not os.path.exists(bed_file):
    print(f"Converting BigBed to BED...")
    !../bigBedToBed {BIGBED_FILE} {bed_file}
    print(f"Converted to {bed_file}")
else:
    print(f"BED file already exists: {bed_file}")

# Load BED file
print(f"\nLoading translons from {bed_file}...")

SCHEMA = [
    ('chrom', 'category'),
    ('start', 'int64'),
    ('end', 'int64'),
    ('name', 'string'),
    ('score', 'int64'),
    ('strand', 'category'),
    ('thickStart', 'int64'),
    ('thickEnd', 'int64'),
    ('itemRgb', 'string'),
    ('blockCount', 'int64'),
    ('blockSizes', 'string'),
    ('blockStarts', 'string'),
    ('orf_id', 'string'),
    ('cds_status', 'category'),
    ('peptide_status', 'category'),
    ('frame', 'string'),
    ('gene_biotype', 'category'),
    ('gene_id', 'string'),
    ('gene_name', 'string'),
    ('transcript_biotype', 'category'),
    ('transcript_type', 'category'),
    ('peptide_seq', 'string'),
    ('transcript_ids', 'string'),
    ('parent_gene', 'string'),
    ('replicated', 'boolean'),
    ('pmids', 'string'),
]

col_names = [c for c, _ in SCHEMA]

bed_df = pd.read_csv(
    bed_file,
    sep="\t",
    header=None,
    names=col_names,
    dtype=str,
    na_values=[".", "NA", "none", "None", ""],
)

for col, dtype in SCHEMA:
    if dtype == "int64":
        bed_df[col] = pd.to_numeric(bed_df[col], errors="raise")
    elif dtype == "int8":
        bed_df[col] = pd.to_numeric(bed_df[col], errors="raise").astype("int8")
    elif dtype == "boolean":
        bed_df[col] = bed_df[col].map({"yes": True, "no": False})
    else:
        bed_df[col] = bed_df[col].astype(dtype)

assert (bed_df["start"] < bed_df["end"]).all(), "start >= end found"
assert bed_df["strand"].isin(["+", "-"]).all(), "invalid strand"
assert (bed_df["blockCount"] > 0).all(), "blockCount <= 0 found"


def parse_blocks(row):
    sizes = [int(x) for x in row.blockSizes.rstrip(",").split(",")]
    starts = [int(x) for x in row.blockStarts.rstrip(",").split(",")]
    frames = [int(x) for x in row.frame.rstrip(",").split(",")]

    return sizes, starts, frames

bed_df[["blockSizes_list", "blockStarts_list", "frames_list"]] = (
    bed_df.apply(parse_blocks, axis=1, result_type="expand")
)

bed_df.head()


BED file already exists: data/Ribo-seq_ORFs.bed

Loading translons from data/Ribo-seq_ORFs.bed...


,chrom,start,end,name,score,strand,thickStart,thickEnd,itemRgb,blockCount,...,transcript_biotype,transcript_type,peptide_seq,transcript_ids,parent_gene,replicated,pmids,blockSizes_list,blockStarts_list,frames_list
0,chr1,826870,829027,c1norep1,0,+,826870,829027,0,2,...,lncRNA,lncRNA,MKKPLPRRSPLLSGTPGSFSPVTMA,ENST00000623808,ENSG00000228794,False,PMID:32139545,"[53, 25]","[0, 2132]","[0, 2]"
1,chr1,829023,829092,c1norep2,0,+,829023,829092,0,1,...,lncRNA,lncRNA,MISAHCDLCFLGSRILLPQPPK,"ENST00000416570,ENST00000445118,ENST0000044897...",ENSG00000228794,False,PMID:27232982,[69],[0],[0]
2,chr1,847671,850321,c1riboseqorf1,0,+,847671,850321,0,2,...,lncRNA,lncRNA,MEFFIPTSVDLKILPLSACLGSAVSSLPWFSDDACNNAMRFAHSWA...,ENST00000448975,ENSG00000228794,True,"PMID:31155234,PMID:32744504","[135, 141]","[0, 2509]","[0, 0]"
3,chr1,847671,852067,c1riboseqorf2,0,+,847671,852067,0,2,...,lncRNA,lncRNA,MEFFIPTSVDLKILPLSACLGSAVSSLPWFSDDACNNAMRFAHSWV...,"ENST00000416570,ENST00000449005,ENST0000065717...",ENSG00000228794,True,"PMID:26687005,PMID:26657557,PMID:31155234,PMID...","[135, 141]","[0, 4255]","[0, 0]"
4,chr1,852070,852699,c1riboseqorf3,0,+,852070,852699,0,2,...,lncRNA,lncRNA,MKVAGAGAVTQPPGMILFKDWI,"ENST00000416570,ENST00000425657,ENST0000044511...",ENSG00000228794,True,"PMID:27232982,PMID:31155234,PMID:32139545,PMID...","[40, 29]","[0, 600]","[0, 1]"


In [53]:

bed_df['translon_id'] = bed_df['name']
print(f"Loaded {len(bed_df):,} translons from BigBed")

# Load transcript annotations
print(f"\nLoading transcript annotations from {TRANSCRIPT_ANNOTATIONS}...")
transcript_df = pd.read_csv(TRANSCRIPT_ANNOTATIONS, sep='\t')
print(f"Loaded {len(transcript_df):,} translon-transcript mappings")

# Create a simple mapping dict from orf_name to transcript
transcript_map = dict(zip(transcript_df['orf_name'], transcript_df['transcript']))

# Add transcript column to bed_df
bed_df['transcript'] = bed_df['translon_id'].map(transcript_map)

# Use bed_df directly (don't merge, to avoid column confusion)
translons = bed_df.copy()

# Calculate exonic length from blockSizes
def calc_exonic_length(row):
    if pd.isna(row['blockSizes']) or row['blockSizes'] == '':
        print(" is na")
        return row['end'] - row['start']
    
    else:
        # blockSizes is comma-separated like "53,25,"
        sizes = [int(x) for x in row['blockSizes'].rstrip(',').split(',') if x.strip()]
        return sum(sizes)


translons['exonic_length'] = translons.apply(calc_exonic_length, axis=1)

print(f"\nDataset summary:")
print(f"  Total translons: {len(translons):,}")
print(f"  With transcript ID: {translons['transcript'].notna().sum():,}")
print(f"  Without transcript ID: {translons['transcript'].isna().sum():,}")
print(f"  Single-exon: {(translons['blockCount'] == 1).sum():,}")
print(f"  Multi-exon: {(translons['blockCount'] > 1).sum():,}")

Loaded 7,264 translons from BigBed

Loading transcript annotations from ../phase1_w_transcript.tsv...
Loaded 7,264 translon-transcript mappings

Dataset summary:
  Total translons: 7,264
  With transcript ID: 7,264
  Without transcript ID: 0
  Single-exon: 4,801
  Multi-exon: 2,463


# Step 3: Helper Functions

In [59]:
def parse_bed_blocks(chrom_start, block_count, block_sizes_str, block_starts_str):
    """
    Parse BED12 block notation into list of (start, end) tuples.
    """
    if pd.isna(block_sizes_str) or pd.isna(block_starts_str):
        return []
    
    try:
        block_sizes = [int(x) for x in str(block_sizes_str).rstrip(',').split(',')]
        block_starts = [int(x) for x in str(block_starts_str).rstrip(',').split(',')]
        
        blocks = []
        for i in range(int(block_count)):
            block_start = chrom_start + block_starts[i]
            block_end = block_start + block_sizes[i]
            blocks.append((block_start, block_end))
        
        return blocks
    except:
        return []


def get_exonic_flanks(translon_chrom, translon_blocks, translon_strand, transcript_id, flank_size=150):
    """
    Get exonic flanking regions by walking along transcript exons.
    Skips introns completely.
    
    Returns:
        upstream_coords: list of (start, end) tuples
        downstream_coords: list of (start, end) tuples
    """
    if not translon_blocks:
        return [], []
    
    translon_min = min(s for s, e in translon_blocks)
    translon_max = max(e for s, e in translon_blocks)
    
    # Get transcript exons
    exons = transcript_exons.get(transcript_id, [])

    # Fallback to genomic flanks if no transcript annotation
    if not exons or pd.isna(transcript_id):
        if translon_strand == '+':
            upstream_coords = [(max(0, translon_min - flank_size), translon_min)]
            downstream_coords = [(translon_max, translon_max + flank_size)]
        else:
            upstream_coords = [(translon_max, translon_max + flank_size)]
            downstream_coords = [(max(0, translon_min - flank_size), translon_min)]
        return upstream_coords, downstream_coords
    
    # Separate exons: before, overlapping, after translon
    before_exons = [e for e in exons if e['end'] <= translon_min]
    after_exons = [e for e in exons if e['start'] >= translon_max]
    translon_exons = [e for e in exons if not (e['end'] <= translon_min or e['start'] >= translon_max)]
    
    strand = exons[0]['strand']
    
    if strand == '+':
        # Upstream: walk backwards collecting exonic bases
        upstream_coords = []
        remaining = flank_size
        
        # Check translon exons for bases before translon
        for exon in reversed(translon_exons):
            if remaining <= 0:
                break
            if exon['start'] < translon_min:
                take_start = max(exon['start'], translon_min - remaining)
                take_end = translon_min
                upstream_coords.insert(0, (take_start, take_end))
                remaining -= (take_end - take_start)
        
        # Walk backwards through before_exons
        for exon in reversed(before_exons):
            if remaining <= 0:
                break
            exon_len = exon['end'] - exon['start']
            if exon_len <= remaining:
                upstream_coords.insert(0, (exon['start'], exon['end']))
                remaining -= exon_len
            else:
                upstream_coords.insert(0, (exon['end'] - remaining, exon['end']))
                remaining = 0
        
        # Extend genomically if needed
        if remaining > 0:
            transcript_start = min(e['start'] for e in exons)
            upstream_coords.insert(0, (max(0, transcript_start - remaining), transcript_start))
        
        # Downstream: walk forwards
        downstream_coords = []
        remaining = flank_size
        
        # Check translon exons for bases after translon
        for exon in translon_exons:
            if remaining <= 0:
                break
            if exon['end'] > translon_max:
                take_start = translon_max
                take_end = min(exon['end'], translon_max + remaining)
                downstream_coords.append((take_start, take_end))
                remaining -= (take_end - take_start)
        
        # Walk forwards through after_exons
        for exon in after_exons:
            if remaining <= 0:
                break
            exon_len = exon['end'] - exon['start']
            if exon_len <= remaining:
                downstream_coords.append((exon['start'], exon['end']))
                remaining -= exon_len
            else:
                downstream_coords.append((exon['start'], exon['start'] + remaining))
                remaining = 0
        
        # Extend genomically if needed
        if remaining > 0:
            transcript_end = max(e['end'] for e in exons)
            downstream_coords.append((transcript_end, transcript_end + remaining))
    
    else:  # strand == '-'
        # For minus strand: upstream (5' of gene) is genomically downstream
        upstream_coords = []
        remaining = flank_size
        
        # Check translon exons for bases after translon (genomically)
        for exon in translon_exons:
            if remaining <= 0:
                break
            if exon['end'] > translon_max:
                take_start = translon_max
                take_end = min(exon['end'], translon_max + remaining)
                upstream_coords.append((take_start, take_end))
                remaining -= (take_end - take_start)
        
        # Walk through after_exons
        for exon in after_exons:
            if remaining <= 0:
                break
            exon_len = exon['end'] - exon['start']
            if exon_len <= remaining:
                upstream_coords.append((exon['start'], exon['end']))
                remaining -= exon_len
            else:
                upstream_coords.append((exon['start'], exon['start'] + remaining))
                remaining = 0
        
        # Extend genomically if needed
        if remaining > 0:
            transcript_end = max(e['end'] for e in exons)
            upstream_coords.append((transcript_end, transcript_end + remaining))
        
        # Downstream (3' of gene) = genomically upstream
        downstream_coords = []
        remaining = flank_size
        
        # Check translon exons for bases before translon
        for exon in reversed(translon_exons):
            if remaining <= 0:
                break
            if exon['start'] < translon_min:
                take_start = max(exon['start'], translon_min - remaining)
                take_end = translon_min
                downstream_coords.insert(0, (take_start, take_end))
                remaining -= (take_end - take_start)
        
        # Walk backwards through before_exons
        for exon in reversed(before_exons):
            if remaining <= 0:
                break
            exon_len = exon['end'] - exon['start']
            if exon_len <= remaining:
                downstream_coords.insert(0, (exon['start'], exon['end']))
                remaining -= exon_len
            else:
                downstream_coords.insert(0, (exon['end'] - remaining, exon['end']))
                remaining = 0
        
        # Extend genomically if needed
        if remaining > 0:
            transcript_start = min(e['start'] for e in exons)
            downstream_coords.insert(0, (max(0, transcript_start - remaining), transcript_start))
    
    return upstream_coords, downstream_coords


def extract_phylop_scores(bw, chrom, coord_list):
    """Extract PhyloP scores from multiple coordinate ranges."""
    all_scores = []
    
    for start, end in coord_list:
        scores = bw.values(chrom, start, end, numpy=True)
        if scores is not None:
            valid_scores = scores[~np.isnan(scores)]
            if len(valid_scores) > 0:
                all_scores.extend(valid_scores)
    
    return np.array(all_scores) if all_scores else np.array([])


def calc_stats(scores):
    """Calculate statistics for PhyloP scores."""
    if len(scores) == 0:
        return {
            'n_bases': 0,
            'mean': np.nan,
            'median': np.nan,
            'std': np.nan,
            'min': np.nan,
            'max': np.nan
        }
    return {
        'n_bases': len(scores),
        'mean': np.mean(scores),
        'median': np.median(scores),
        'std': np.std(scores),
        'min': np.min(scores),
        'max': np.max(scores)
    }

print("Helper functions defined")

Helper functions defined


# Step 4: Caluclate Scores

In [60]:
print(f"Opening PhyloP 470way: {PHYLOP_470WAY}")
bw = pyBigWig.open(PHYLOP_470WAY)

results = []

for idx, row in translons.iterrows():
    if idx % 1000 == 0:
        print(f"Processing {idx+1:,}/{len(translons):,}...")
    
    translon_id = row['translon_id']
    chrom = row['chrom']
    start = row['start']
    end = row['end']
    strand = row['strand']
    transcript_id = row.get('transcript', None)
    
    # Parse translon blocks
    block_count = row.get('blockCount', 1)
    block_sizes = row.get('blockSizes', '')
    block_starts = row.get('blockStarts', '')
    
    translon_blocks = parse_bed_blocks(start, block_count, block_sizes, block_starts)
    if not translon_blocks:
        translon_blocks = [(start, end)]

    # Get exonic flanks
    upstream_coords, downstream_coords = get_exonic_flanks(
        chrom, translon_blocks, strand, transcript_id, FLANK_SIZE
    )
    
    # Extract PhyloP scores
    feature_scores = extract_phylop_scores(bw, chrom, translon_blocks)
    upstream_scores = extract_phylop_scores(bw, chrom, upstream_coords)
    downstream_scores = extract_phylop_scores(bw, chrom, downstream_coords)
    
    # Calculate statistics
    feature_stats = calc_stats(feature_scores)
    upstream_stats = calc_stats(upstream_scores)
    downstream_stats = calc_stats(downstream_scores)
    
    # Derived metrics
    feature_mean = feature_stats['mean']
    upstream_mean = upstream_stats['mean']
    downstream_mean = downstream_stats['mean']
    
    if not np.isnan(feature_mean) and not np.isnan(upstream_mean) and not np.isnan(downstream_mean):
        conservation_specificity = feature_mean - max(upstream_mean, downstream_mean)
    else:
        conservation_specificity = np.nan
    
    # Store result
    result = {
        'translon_id': translon_id,
        'chrom': chrom,
        'start': start,
        'end': end,
        'strand': strand,
        'transcript_id': transcript_id if pd.notna(transcript_id) else 'no_annotation',
        'exonic_length': row['exonic_length'],
        'blockCount': block_count,
        'flank_size': FLANK_SIZE,
        'flank_type': 'exonic',
        
        'feature_n_bases': feature_stats['n_bases'],
        'feature_mean': feature_stats['mean'],
        'feature_median': feature_stats['median'],
        'feature_std': feature_stats['std'],
        'feature_min': feature_stats['min'],
        'feature_max': feature_stats['max'],
        
        'upstream_n_bases': upstream_stats['n_bases'],
        'upstream_mean': upstream_stats['mean'],
        'upstream_median': upstream_stats['median'],
        'upstream_std': upstream_stats['std'],
        'upstream_min': upstream_stats['min'],
        'upstream_max': upstream_stats['max'],
        
        'downstream_n_bases': downstream_stats['n_bases'],
        'downstream_mean': downstream_stats['mean'],
        'downstream_median': downstream_stats['median'],
        'downstream_std': downstream_stats['std'],
        'downstream_min': downstream_stats['min'],
        'downstream_max': downstream_stats['max'],
        
        'conservation_specificity': conservation_specificity,
    }
    
    results.append(result)

bw.close()

df_results = pd.DataFrame(results)

print(f"\nCompleted processing {len(df_results):,} translons")
print(f"\nMean PhyloP scores:")
print(f"  Feature:    {df_results['feature_mean'].mean():6.3f}")
print(f"  Upstream:   {df_results['upstream_mean'].mean():6.3f}")
print(f"  Downstream: {df_results['downstream_mean'].mean():6.3f}")

Opening PhyloP 470way: data/phylop/hg38.phyloP470way.bw
Processing 1/7,264...
Processing 1,001/7,264...
Processing 2,001/7,264...
Processing 3,001/7,264...
Processing 4,001/7,264...
Processing 5,001/7,264...
Processing 6,001/7,264...
Processing 7,001/7,264...

Completed processing 7,264 translons

Mean PhyloP scores:
  Feature:     1.560
  Upstream:    1.320
  Downstream:  1.989


# Step 5: identify CDS overlaps

In [61]:
print("Extracting CDS features from GENCODE GTF...")

cds_records = []

with gzip.open(GENCODE_GTF, 'rt') as f:
    for line in f:
        if line.startswith('#'):
            continue
        
        fields = line.strip().split('\t')
        if len(fields) < 9 or fields[2] != 'CDS':
            continue
        
        chrom = fields[0]
        start = int(fields[3]) - 1
        end = int(fields[4])
        strand = fields[6]
        
        attrs = {}
        for attr in fields[8].split(';'):
            attr = attr.strip()
            if attr:
                parts = attr.split(' ', 1)
                if len(parts) == 2:
                    key, val = parts
                    attrs[key] = val.strip('"')
        
        cds_records.append({
            'Chromosome': chrom,
            'Start': start,
            'End': end,
            'Strand': strand,
            'gene_name': attrs.get('gene_name', ''),
        })

print(f"Found {len(cds_records):,} CDS features")

# Create PyRanges
cds_pr = pr.PyRanges(pd.DataFrame(cds_records))

translon_records = []
for _, row in df_results.iterrows():
    translon_records.append({
        'Chromosome': row['chrom'],
        'Start': row['start'],
        'End': row['end'],
        'Strand': row['strand'],
        'translon_id': row['translon_id']
    })

translon_pr = pr.PyRanges(pd.DataFrame(translon_records))

print("Finding overlaps...")
overlaps = translon_pr.join(cds_pr, how='left', suffix='_cds')

# Process overlaps and calculate proportion
overlap_results = []
for idx, row in df_results.iterrows():
    translon_id = row['translon_id']
    translon_len = row['end'] - row['start']
    
    translon_overlaps = overlaps.df[
        (overlaps.df['Chromosome'] == row['chrom']) &
        (overlaps.df['Start'] == row['start']) &
        (overlaps.df['End'] == row['end'])
    ]
    
    if len(translon_overlaps) > 0 and 'Start_cds' in translon_overlaps.columns:
        has_overlap = translon_overlaps['Start_cds'].notna().any()
        
        if has_overlap:
            overlapping_genes = translon_overlaps['gene_name'].dropna().unique()
            same_strand = (translon_overlaps['Strand'] == translon_overlaps['Strand_cds']).any()
            
            # Calculate overlap proportion
            overlap_bases = 0
            for _, overlap_row in translon_overlaps.iterrows():
                if pd.notna(overlap_row['Start_cds']):
                    overlap_start = max(row['start'], overlap_row['Start_cds'])
                    overlap_end = min(row['end'], overlap_row['End_cds'])
                    if overlap_end > overlap_start:
                        overlap_bases += (overlap_end - overlap_start)
            
            overlap_proportion = min(1.0, overlap_bases / translon_len) if translon_len > 0 else 0.0
            
            overlap_results.append({
                'translon_id': translon_id,
                'has_cds_overlap': True,
                'same_strand': same_strand,
                'cds_overlap_proportion': overlap_proportion,
                'overlapping_genes': ','.join(overlapping_genes[:5]) if len(overlapping_genes) > 0 else '-1'
            })
        else:
            overlap_results.append({
                'translon_id': translon_id,
                'has_cds_overlap': False,
                'same_strand': False,
                'cds_overlap_proportion': 0.0,
                'overlapping_genes': '-1'
            })
    else:
        overlap_results.append({
            'translon_id': translon_id,
            'has_cds_overlap': False,
            'same_strand': False,
            'cds_overlap_proportion': 0.0,
            'overlapping_genes': '-1'
        })

overlap_df = pd.DataFrame(overlap_results)

# Correct for PyRanges -1 sentinel
overlap_df['has_cds_overlap'] = overlap_df['overlapping_genes'] != '-1'

# Merge with results
df_results = df_results.merge(overlap_df, on='translon_id', how='left')

print(f"\nCDS overlap results:")
print(f"  With CDS overlap: {df_results['has_cds_overlap'].sum():,} ({100*df_results['has_cds_overlap'].sum()/len(df_results):.1f}%)")
print(f"  NO CDS overlap:   {(~df_results['has_cds_overlap']).sum():,} ({100*(~df_results['has_cds_overlap']).sum()/len(df_results):.1f}%)")
print(f"\nMean CDS overlap proportion: {df_results[df_results['has_cds_overlap']]['cds_overlap_proportion'].mean():.2%}")

Extracting CDS features from GENCODE GTF...
Found 899,146 CDS features
Finding overlaps...

CDS overlap results:
  With CDS overlap: 1,970 (27.1%)
  NO CDS overlap:   5,294 (72.9%)

Mean CDS overlap proportion: 50.80%


In [73]:
print("="*80)
print("CLASSIFYING TRANSLONS")
print("="*80)

# 1. Calculate max_flank for ALL translons
df_results['max_flank'] = df_results[['upstream_mean', 'downstream_mean']].max(axis=1)

# 2. Apply Classification Logic
def classify_translon(row):
    # 1. Non-conserved (Hard Filter)
    if row['feature_mean'] <= 0.5:
        return 'non_conserved'
    
    # 2. Regional Conservation (High context signal)
    # Priorities shape: if flanks are high, it is regional context
    if row['max_flank'] > FLANK_THRESHOLD:
        return 'regional_conservation'
    
    # 3. CDS Associated (Known Genes)
    # If not regional, check if it matches known CDS
    if row['has_cds_overlap']:
        return 'cds_associated'
    
    # 4. Weakly conserved
    # Only captures features that are weak AND have low flanks (isolated weak signals)
    if row['feature_mean'] <= CONSERVATION_THRESHOLD:
        return 'weakly_conserved'
    
    # 5. Restricted Candidate
    # Strong feature + Low flanks + No CDS
    return 'restricted_candidate'

df_results['classification'] = df_results.apply(classify_translon, axis=1)

# 3. Print Summary
counts = df_results['classification'].value_counts()
percentages = df_results['classification'].value_counts(normalize=True) * 100

print(f"\nClassification Results (Total: {len(df_results):,}):")
for cat in ['restricted_candidate', 'regional_conservation', 'cds_associated', 'weakly_conserved', 'non_conserved']:
    count = counts.get(cat, 0)
    pct = percentages.get(cat, 0)
    print(f"  {cat:<25}: {count:>6,} ({pct:>5.1f}%)")

restricted = df_results[df_results['classification'] == 'restricted_candidate']
print(f"\nFound {len(restricted)} restricted candidates for detailed analysis.")

CLASSIFYING TRANSLONS

Classification Results (Total: 7,264):
  restricted_candidate     :     43 (  0.6%)
  regional_conservation    :  3,812 ( 52.5%)
  cds_associated           :     79 (  1.1%)
  weakly_conserved         :    228 (  3.1%)
  non_conserved            :  3,102 ( 42.7%)

Found 43 restricted candidates for detailed analysis.


In [74]:
# Save comprehensive results
comprehensive_output = OUTPUT_DIR / 'translon_phylop_v3_with_cds_overlap.tsv'
df_results.to_csv(comprehensive_output, sep='\t', index=False)

print(f"Saved comprehensive results: {comprehensive_output}")
print(f"  Rows: {len(df_results):,}")
print(f"  Columns: {len(df_results.columns)}")

Saved comprehensive results: ../results/phylop/translon_phylop_v3_with_cds_overlap.tsv
  Rows: 7,264
  Columns: 36


In [75]:
print("Generating final output files...\n")

# ============================================================================
# File 1: Classified Story File
# ============================================================================

key_columns = [
    'translon_id', 'classification', 'chrom', 'start', 'end', 'exonic_length', 
    'feature_mean', 'max_flank', 'upstream_mean', 'downstream_mean',
    'conservation_specificity', 'has_cds_overlap', 'overlapping_genes'
]

# Proper sorting order
category_order = {
    'restricted_candidate': 0,
    'regional_conservation': 1,
    'cds_associated': 2,
    'weakly_conserved': 3,
    'non_conserved': 4
}

df_results['sort_order'] = df_results['classification'].map(category_order)
final_basic = df_results.sort_values(['sort_order', 'conservation_specificity'], ascending=[True, False]).copy()

output_basic = OUTPUT_DIR / 'v3_translon_conservation_story_classified.tsv'
final_basic[key_columns].to_csv(output_basic, sep='\t', index=False)

print(f"1. CLASSIFIED STORY FILE: {output_basic}")
print(f"   - Rows sorted by classification priority, then specificity\n")

# Update comprehensive results with new columns
comprehensive_output = OUTPUT_DIR / 'translon_phylop_v3_with_cds_overlap.tsv'
df_results.to_csv(comprehensive_output, sep='\t', index=False)
print(f"2. UPDATED COMPREHENSIVE RESULTS: {comprehensive_output}")


Generating final output files...

1. CLASSIFIED STORY FILE: ../results/phylop/v3_translon_conservation_story_classified.tsv
   - Rows sorted by classification priority, then specificity

2. UPDATED COMPREHENSIVE RESULTS: ../results/phylop/translon_phylop_v3_with_cds_overlap.tsv


In [76]:
print("\n" + "="*80)
print("FINAL SUMMARY")
print("="*80)

print(f"\nTotal translons: {len(df_results):,}")
print("\nCategories:")
print(df_results['classification'].value_counts())

print(f"\nOutput file: {output_basic}")


FINAL SUMMARY

Total translons: 7,264

Categories:
regional_conservation    3812
non_conserved            3102
weakly_conserved          228
cds_associated             79
restricted_candidate       43
Name: classification, dtype: int64

Output file: ../results/phylop/v3_translon_conservation_story_classified.tsv
